# TPC-H Data Realism (library notebook)

Volume-preserving reshaping for the **initial staging load** (`batch_id = 1`).
Invoked via `%run ./tpch_data_realism` from `setup_catalogs_and_staging.ipynb`.

Tiers (see `scratch/plans/DATA_REALISM_PLAN.md`):
1. Temporal — growth + seasonality via weighted date remapping (orders + lineitem ship/commit dates)
2. Supplier — Pareto-style value skew on `l_extendedprice`
3. Category — relabel `o_orderpriority` and `l_shipmode` to non-uniform shares
4. Market segment — relabel customer `mktseg` to non-uniform shares
5. Region — relabel customer `nat_id` so geography skews toward target region shares

Run 2/3 narrative keys (customers 1–3, suppliers 1–3) are **pinned** — excluded from remap/relabel.

Note: two always-on, batch-independent adjustments live in `initialize.ipynb` (applied to every
batch, not gated by the realism toggle): the **unit-price reduction** (`PRICE_SCALE`, so total net
sales land near ~$800M) and the **ship-mode fulfillment SLA** (`receipt_date = ship_date +
transit(mode)`, so on-time delivery varies by ship mode).

In [ ]:
import time
from datetime import date
from pyspark.sql import Window

dbutils.widgets.dropdown("data_realism_enabled", "true", ["true", "false"], "Apply data realism on initial staging load")
data_realism_enabled = dbutils.widgets.get("data_realism_enabled").lower() == "true"

# Keys used by Run 2/3 incremental demos — keep their business timeline / values stable.
PINNED_CUSTOMER_IDS = (1, 2, 3)
PINNED_SUPPLIER_IDS = (1, 2, 3)

ORDER_PRIORITIES = ["1-URGENT", "2-HIGH", "3-MEDIUM", "4-NOT SPECIFIED", "5-LOW"]
ORDER_PRIORITY_CUM_PCT = [8, 20, 45, 75, 100]  # minority URGENT/HIGH

SHIP_MODES = ["AIR", "REG AIR", "TRUCK", "SHIP", "RAIL", "MAIL", "FOB"]
SHIP_MODE_CUM_PCT = [25, 40, 60, 75, 85, 93, 100]  # AIR + TRUCK dominant

# Market segment (customer) — non-uniform share instead of even 20% each.
MKT_SEGMENTS = ["BUILDING", "AUTOMOBILE", "MACHINERY", "HOUSEHOLD", "FURNITURE"]
MKT_SEGMENT_CUM_PCT = [30, 55, 75, 90, 100]

# Region skew (customer geography) via nation relabel. Standard TPC-H nation_key -> region_key,
# and target region shares (EUROPE + ASIA lead; AFRICA smallest) instead of even 20% each.
NATION_TO_REGION = {
    0: 0, 5: 0, 14: 0, 15: 0, 16: 0,      # AFRICA
    1: 1, 2: 1, 3: 1, 17: 1, 24: 1,       # AMERICA
    8: 2, 9: 2, 12: 2, 18: 2, 21: 2,      # ASIA
    6: 3, 7: 3, 19: 3, 22: 3, 23: 3,      # EUROPE
    4: 4, 10: 4, 11: 4, 13: 4, 20: 4,     # MIDDLE EAST
}
REGION_SHARES = {3: 0.28, 2: 0.24, 1: 0.22, 4: 0.14, 0: 0.12}  # EUROPE, ASIA, AMERICA, MIDEAST, AFRICA


class RealismTimer:
    """Accumulate per-phase seconds for setup benchmarking."""

    def __init__(self):
        self.phases = {}

    def add(self, phase, seconds):
        self.phases[phase] = self.phases.get(phase, 0.0) + seconds

    def summary_lines(self):
        if not self.phases:
            return ["  (no realism timing recorded)"]
        total = sum(self.phases.values())
        lines = [f"  realism total: {total:.2f}s"]
        for name, secs in sorted(self.phases.items()):
            lines.append(f"    {name}: {secs:.2f}s")
        return lines


realism_timer = RealismTimer()
realism_context = None


def _month_slots():
    """Target month buckets with YoY growth x seasonality weights (1992-01 .. 1998-08)."""
    seasonal = {1: 0.85, 2: 0.88, 3: 0.92, 4: 0.95, 5: 1.00, 6: 1.02,
                7: 1.00, 8: 0.98, 9: 1.05, 10: 1.08, 11: 1.12, 12: 1.18}
    slots = []
    slot_id = 1
    for year in range(1992, 1999):
        for month in range(1, 13):
            if year == 1998 and month > 8:
                break
            growth = 1.0 + 0.08 * (year - 1992)
            weight = growth * seasonal[month]
            slots.append((slot_id, date(year, month, 1), float(weight)))
            slot_id += 1
    return slots


def _relabel_by_percentile(df, hash_cols, labels, cum_pct, out_col):
    """Map ntile(100) buckets to categorical labels (volume-preserving relabel)."""
    hash_expr = F.hash(*[F.col(c) for c in hash_cols])
    bucket = F.ntile(100).over(Window.orderBy(hash_expr))
    expr = F.lit(labels[-1])
    for label, upper in reversed(list(zip(labels, cum_pct))):
        expr = F.when(bucket <= upper, F.lit(label)).otherwise(expr)
    return df.withColumn(out_col, expr)


def _build_nation_target_df(spark):
    """Cumulative [lo, hi) bounds mapping a uniform [0,1) position to a target nation_key, so the
    customer base skews toward the target REGION_SHARES (nations equal-weighted within a region)."""
    region_nations = {}
    for nk, rk in NATION_TO_REGION.items():
        region_nations.setdefault(rk, []).append(nk)
    rows, acc = [], 0.0
    for rk, share in REGION_SHARES.items():
        nats = sorted(region_nations[rk])
        per_nation = share / len(nats)
        for nk in nats:
            lo = acc
            acc += per_nation
            rows.append((int(nk), float(lo), float(acc)))
    # widen the final upper bound so percent_rank()==1.0 lands in the last nation slot
    _nk, _lo, _ = rows[-1]
    rows[-1] = (_nk, _lo, 1.01)
    return spark.createDataFrame(rows, "target_nat int, nlo double, nhi double")


def build_realism_context(spark, sample_source_schema):
    """Precompute broadcast maps once per setup run. Returns dict or None if disabled."""
    global realism_context
    if not data_realism_enabled:
        realism_context = None
        return None

    t0 = time.perf_counter()

    # --- Tier 1: orderkey -> date delta (days), weighted month allocation ---
    # Build cumulative [lo, hi) weight bounds per month so each slot receives
    # volume proportional to growth x seasonality (not an equal ntile split).
    slots = _month_slots()
    n_slots = len(slots)
    total_w = sum(w for _, _, w in slots)
    slot_rows, cum = [], 0.0
    for sid, month_start, w in slots:
        lo = cum / total_w
        cum += w
        hi = cum / total_w
        slot_rows.append((sid, month_start, float(lo), float(hi)))
    # widen the final upper bound so percent_rank()==1.0 lands in the last slot
    _sid, _ms, _lo, _ = slot_rows[-1]
    slot_rows[-1] = (_sid, _ms, _lo, 1.01)
    slots_df = spark.createDataFrame(
        slot_rows, "slot_id int, month_start date, lo double, hi double"
    )

    # Uniform position in [0,1] by hashed orderkey, then bucket by cumulative
    # weight via a broadcast range join (80 slot rows).
    orders = spark.table(f"{sample_source_schema}.orders")
    orders = orders.withColumn(
        "_u", F.percent_rank().over(Window.orderBy(F.hash("o_orderkey")))
    )
    orders = orders.join(
        F.broadcast(slots_df),
        (F.col("_u") >= F.col("lo")) & (F.col("_u") < F.col("hi")),
        "inner",
    )
    orders = orders.withColumn(
        "_day_offset",
        F.pmod(F.hash(F.col("o_orderkey")), F.dayofmonth(F.last_day("month_start"))),
    )
    orders = orders.withColumn(
        "_new_orderdate",
        F.when(
            F.col("o_custkey").isin(list(PINNED_CUSTOMER_IDS)),
            F.col("o_orderdate"),
        ).otherwise(F.date_add(F.col("month_start"), F.col("_day_offset"))),
    )
    order_delta = orders.select(
        F.col("o_orderkey"),
        F.datediff(F.col("_new_orderdate"), F.col("o_orderdate")).alias("delta_days"),
    )

    # --- Tier 2: supplier value weights (Pareto-ish, deterministic by key) ---
    # Rank-based Zipf weighting: rank suppliers by hashed key, then weight ~ 1/rank^0.9. This yields
    # a clear #1 >> #2 >> #3 leaderboard that decays smoothly into a long tail, instead of the plateau
    # a bucketed power law produced (where the top ~40 suppliers tied at the max weight).
    supplier = spark.table(f"{sample_source_schema}.supplier")
    supplier = supplier.withColumn(
        "_supp_rank", F.row_number().over(Window.orderBy(F.hash("s_suppkey")))
    )
    supplier_weights = supplier.select(
        F.col("s_suppkey"),
        F.when(
            F.col("s_suppkey").isin(list(PINNED_SUPPLIER_IDS)),
            F.lit(1.0),
        )
        .otherwise(
            F.lit(0.5) + F.lit(2800.0) / F.pow(F.col("_supp_rank").cast("double"), F.lit(0.9))
        )
        .alias("supplier_value_weight"),
    )

    realism_context = {
        "order_delta": order_delta,
        "supplier_weights": supplier_weights,
    }
    realism_timer.add("build_context", time.perf_counter() - t0)
    print(f"Data realism: context built ({realism_timer.phases['build_context']:.2f}s, {n_slots} month slots)")
    return realism_context


def apply_staging_realism(df, entity):
    """Apply tiered reshaping to a staging dataframe before metadata/landing."""
    if not data_realism_enabled or realism_context is None:
        return df

    t0 = time.perf_counter()
    ctx = realism_context

    if entity == "orders":
        df = df.join(F.broadcast(ctx["order_delta"]), "o_orderkey", "left")
        df = df.withColumn("o_orderdate", F.date_add(F.col("o_orderdate"), F.col("delta_days")))
        df = df.drop("delta_days")
        df = _relabel_by_percentile(
            df, ["o_orderkey"], ORDER_PRIORITIES, ORDER_PRIORITY_CUM_PCT, "o_orderpriority"
        )

    elif entity == "lineitem":
        order_delta = ctx["order_delta"].withColumnRenamed("o_orderkey", "l_orderkey")
        df = df.join(F.broadcast(order_delta), "l_orderkey", "left")
        # receipt_date is re-derived from ship_date + ship-mode SLA downstream (apply_ship_mode_sla),
        # so only the ship and commit dates are remapped here.
        for col_name in ("l_shipdate", "l_commitdate"):
            df = df.withColumn(col_name, F.date_add(F.col(col_name), F.col("delta_days")))
        df = df.drop("delta_days")
        supp_w = ctx["supplier_weights"].withColumnRenamed("s_suppkey", "l_suppkey")
        df = df.join(F.broadcast(supp_w), "l_suppkey", "left")
        df = df.withColumn(
            "l_extendedprice",
            (F.col("l_extendedprice") * F.col("supplier_value_weight")).cast("decimal(18,2)"),
        )
        df = df.drop("supplier_value_weight")
        df = _relabel_by_percentile(
            df, ["l_orderkey", "l_linenumber"], SHIP_MODES, SHIP_MODE_CUM_PCT, "l_shipmode"
        )

    elif entity == "customer":
        # Skew market segment away from an even 20% split (pinned customers keep their segment).
        df = df.withColumn("_orig_mktseg", F.col("mktseg"))
        df = _relabel_by_percentile(
            df, ["customer_id"], MKT_SEGMENTS, MKT_SEGMENT_CUM_PCT, "mktseg"
        )
        df = df.withColumn(
            "mktseg",
            F.when(F.col("customer_id").isin(list(PINNED_CUSTOMER_IDS)), F.col("_orig_mktseg"))
            .otherwise(F.col("mktseg")),
        ).drop("_orig_mktseg")

    elif entity == "customer_address":
        # Skew customer geography (region via nation) away from an even split; pinned keep nation.
        nat_df = _build_nation_target_df(spark)
        df = df.withColumn("_u", F.percent_rank().over(Window.orderBy(F.hash("customer_id"))))
        df = df.join(
            F.broadcast(nat_df),
            (F.col("_u") >= F.col("nlo")) & (F.col("_u") < F.col("nhi")),
            "left",
        )
        df = df.withColumn(
            "nat_id",
            F.when(F.col("customer_id").isin(list(PINNED_CUSTOMER_IDS)), F.col("nat_id"))
            .otherwise(F.col("target_nat")),
        ).drop("_u", "nlo", "nhi", "target_nat")

    realism_timer.add(f"transform_{entity}", time.perf_counter() - t0)
    return df


def print_realism_timing_report(setup_elapsed_sec=None):
    """Print realism overhead vs total setup cell duration."""
    if not data_realism_enabled:
        print("Data realism: disabled (widget data_realism_enabled=false)")
        return
    print("\n=== Data realism timing ===")
    for line in realism_timer.summary_lines():
        print(line)
    if setup_elapsed_sec is not None:
        realism_total = sum(realism_timer.phases.values())
        pct = 100.0 * realism_total / setup_elapsed_sec if setup_elapsed_sec else 0
        print(f"  setup load cell total: {setup_elapsed_sec:.2f}s")
        print(f"  realism as % of setup load: {pct:.1f}%")
    print("(Re-run with data_realism_enabled=false on the same cluster to compare.)")